# CitationGuard AI — Phase 5: Scientific Transformer Benchmark (Google Colab GPU)

**Project**: CitationGuard AI — Evidence-Aware Citation Verification for Detecting Unsupported Academic Claims  
**Model**: `microsoft/BiomedNLP-PubMedBERT-base-uncased-abstract`  
**Primary Task**: 3-Class Citation Verification (`SUPPORTED`, `CONTRADICTED`, `NOT_ENOUGH_EVIDENCE`)  
**Primary Evaluation Metric**: Macro-F1 (with strict percentage points comparison against Phase 4 TF-IDF baseline)  
**Hardware Target**: NVIDIA T4 / V100 / A100 GPU (Google Colab)

---
### Experimental Protocol Overview
1. **Environment & GPU Audit**: Automatic verification of CUDA GPU accelerator.
2. **Data Integrity Pre-Flight Check**: Strict verification of the frozen 1,754 records (1,226 TRAIN / 263 DEV / 265 TEST).
3. **Experiment 1 (Primary)**: PubMedBERT fine-tuning on **Strict Evidence** (`Claim + Title + Strict Evidence`).
4. **Experiment 2 (Ablation)**: PubMedBERT fine-tuning on **Abstract Fallback Context** (`Claim + Title + Abstract Fallback`).
5. **Stratified Subsets**: Independent evaluation on Combined TEST, Natural-Only TEST, Synthetic-Negation TEST, and Source-wise splits (SCitance vs SciFact).

## Step 1: Install Dependencies & Check GPU Accelerator

In [ ]:
# Install required packages
!pip install -q transformers>=4.35.0 accelerate>=0.24.0 scikit-learn>=1.2.0 matplotlib>=3.6.0

import sys
import torch
import transformers
import sklearn

print("=" * 65)
print("ENVIRONMENT AUDIT:")
print(f"Python Version:       {sys.version.split()[0]}")
print(f"PyTorch Version:      {torch.__version__}")
print(f"Transformers Version: {transformers.__version__}")
print(f"Scikit-Learn Version: {sklearn.__version__}")
print(f"CUDA Available:       {torch.cuda.is_available()}")

if not torch.cuda.is_available():
    raise RuntimeError(
        "\n[CRITICAL WARNING] No GPU detected!\n"
        "Please enable a GPU runtime before training:\n"
        "Runtime -> Change runtime type -> Hardware accelerator -> T4 GPU.\n"
        "Training PubMedBERT on CPU is not permitted for the official benchmark."
    )
else:
    gpu_name = torch.cuda.get_device_name(0)
    vram_gb = torch.cuda.get_device_properties(0).total_memory / (1024 ** 3)
    print(f"GPU Accelerator:      {gpu_name}")
    print(f"Total GPU VRAM:       {vram_gb:.2f} GB")
    print(f"CUDA Version:         {torch.version.cuda}")
print("=" * 65)

## Step 2: Dataset Loading (Google Drive or Direct Upload)

The dataset must be the exact frozen file: `citationguard_verification.jsonl` ($N=1,754$).

In [ ]:
import os
from google.colab import drive, files

# Set desired data directory
DATA_DIR = "./data"
os.makedirs(DATA_DIR, exist_ok=True)
JSONL_PATH = os.path.join(DATA_DIR, "citationguard_verification.jsonl")

# Choose loading method:
USE_GOOGLE_DRIVE = False  # Set to True if dataset is stored in Google Drive

if USE_GOOGLE_DRIVE:
    drive.mount('/content/drive')
    # Example path: update to your Drive location
    DRIVE_SRC = "/content/drive/MyDrive/CitationGuard/citationguard_verification.jsonl"
    if os.path.exists(DRIVE_SRC):
        import shutil
        shutil.copy(DRIVE_SRC, JSONL_PATH)
        print(f"Loaded dataset from Google Drive: {JSONL_PATH}")
    else:
        print(f"File not found at {DRIVE_SRC}. Please check path or use direct upload.")

if not os.path.exists(JSONL_PATH):
    print("\nPlease upload 'citationguard_verification.jsonl' directly:")
    uploaded = files.upload()
    for fname in uploaded:
        if fname.endswith(".jsonl"):
            os.rename(fname, JSONL_PATH)
            print(f"Saved uploaded file to {JSONL_PATH}")
            break

print(f"Dataset ready at: {JSONL_PATH} ({os.path.getsize(JSONL_PATH):,} bytes)")

## Step 3: Run Pre-Flight Data Integrity Audit (Rule 19)

In [ ]:
import json
from collections import Counter

with open(JSONL_PATH, "r", encoding="utf-8") as f:
    records = [json.loads(line) for line in f if line.strip()]

total = len(records)
splits = Counter(r["guard_split"] for r in records)
labels = Counter(r["verification_label"] for r in records)
sources = Counter(r["source_dataset"] for r in records)

print("=" * 65)
print("PRE-FLIGHT DATA INTEGRITY CHECK (Rule 19):")
print(f"Total Records: {total} (Expected: 1754)")
print(f"Splits:        {dict(splits)} (Expected: TRAIN=1226, DEV=263, TEST=265)")
print(f"Labels:        {dict(labels)} (Expected: 3 classes)")
print(f"Sources:       {dict(sources)}")
print(f"MSVEC Present: {'MSVEC' in sources}")
print("=" * 65)

assert total == 1754, f"Total mismatch: {total}"
assert splits["TRAIN"] == 1226, f"TRAIN split mismatch: {splits['TRAIN']}"
assert splits["DEV"] == 263, f"DEV split mismatch: {splits['DEV']}"
assert splits["TEST"] == 265, f"TEST split mismatch: {splits['TEST']}"
assert set(labels.keys()) == {"SUPPORTED", "CONTRADICTED", "NOT_ENOUGH_EVIDENCE"}
assert "MSVEC" not in sources
print("\n[PASSED] ALL DATA INTEGRITY CHECKS SATISFIED! Ready for training.")

## Step 4: Token Length Distribution Audit (Step 6)

In [ ]:
from transformers import AutoTokenizer
import numpy as np

MODEL_NAME = "microsoft/BiomedNLP-PubMedBERT-base-uncased-abstract"
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

lengths_strict = []
lengths_fallback = []

for r in records:
    claim = r.get("claim_text", "")
    title = r.get("paper_title", "")
    ev_strict = r.get("evidence_text") or "[NO_EVIDENCE_PROVIDED]"
    ev_fallback = r.get("evidence_text") or r.get("paper_abstract") or "[NO_EVIDENCE_PROVIDED]"
    
    text_s = f"Claim:\n{claim}\n\nTitle:\n{title}\n\nEvidence:\n{ev_strict}"
    text_f = f"Claim:\n{claim}\n\nTitle:\n{title}\n\nEvidence:\n{ev_fallback}"
    
    lengths_strict.append(len(tokenizer.encode(text_s, truncation=False)))
    lengths_fallback.append(len(tokenizer.encode(text_f, truncation=False)))

print("TOKEN LENGTH AUDIT:")
print(f"Strict Context   -> Mean: {np.mean(lengths_strict):.1f}, Median: {np.median(lengths_strict):.1f}, 95th: {np.percentile(lengths_strict, 95):.1f}, Max: {np.max(lengths_strict)}")
print(f"Fallback Context -> Mean: {np.mean(lengths_fallback):.1f}, Median: {np.median(lengths_fallback):.1f}, 95th: {np.percentile(lengths_fallback, 95):.1f}, Max: {np.max(lengths_fallback)}")

trunc_strict_512 = (np.array(lengths_strict) > 512).mean() * 100
trunc_fallback_512 = (np.array(lengths_fallback) > 512).mean() * 100
print(f"\nTruncation Rate at max_seq_length=512:")
print(f"Strict Evidence:   {trunc_strict_512:.2f}%")
print(f"Abstract Fallback: {trunc_fallback_512:.2f}%")

## Step 5: Execute PubMedBERT GPU Training Pipeline

Runs both:
1. **Experiment 1 (Strict Evidence)**: Claim + Title + Strict Evidence
2. **Experiment 2 (Abstract Fallback)**: Claim + Title + Abstract Fallback Context

With balanced class weights, AdamW (`lr=2e-5`), batch size 8 with gradient accumulation 2 (effective batch size 16), max length 512, FP16 mixed precision, per-epoch DEV validation, and comprehensive multi-subset test evaluation.

In [ ]:
# Execute the Colab GPU training script
!python train_pubmedbert_colab.py \
    --data_path ./data/citationguard_verification.jsonl \
    --output_dir ./phase5_output \
    --epochs 3 \
    --batch_size 8 \
    --grad_accum 2 \
    --lr 2e-5 \
    --max_length 512

## Step 6: Inline Display of Training Curves & Confusion Matrices

In [ ]:
from IPython.display import Image, display

print("=== EXPERIMENT 1: STRICT EVIDENCE FIGURES ===")
display(Image("./phase5_output/figures/training_curves_pubmedbert_strict.png"))
display(Image("./phase5_output/figures/confusion_matrix_pubmedbert_strict.png"))

print("\n=== EXPERIMENT 2: ABSTRACT FALLBACK FIGURES ===")
display(Image("./phase5_output/figures/training_curves_pubmedbert_fallback.png"))
display(Image("./phase5_output/figures/confusion_matrix_pubmedbert_fallback.png"))

## Step 7: View Final Benchmark Comparison & Scientific Report

In [ ]:
with open("./phase5_output/reports/PHASE5_PUBMEDBERT_REPORT.md", "r", encoding="utf-8") as f:
    report_md = f.read()

from IPython.display import Markdown
display(Markdown(report_md))

## Step 8: Download / Archive Results

Optionally bundle all output artifacts into a single zip archive for local storage.

In [ ]:
!zip -r phase5_pubmedbert_results.zip ./phase5_output
from google.colab import files
files.download("phase5_pubmedbert_results.zip")
print("Downloaded phase5_pubmedbert_results.zip!")